In [1]:
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
df = pd.read_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), parse_dates=["Order Date"])

BG = "#1A1E43"; PANEL = "#20264F"; TEXT = "#FFFFFF"; TICK = "#9BA0BC"
BLUE, SKY, TEAL, GOLD, RED = "#0070C0", "#29ABE2", "#11A7AD", "#F5C353", "#E74E69"

chart_layout = dict(
    plot_bgcolor=PANEL, paper_bgcolor=PANEL,
    font=dict(color=TEXT), margin=dict(l=40, r=20, t=40, b=30),
    height=350)

app = Dash(__name__)

app.layout = html.Div(style={'backgroundColor': BG, 'minHeight': '100vh', 'padding': '20px', 'fontFamily': 'Microsoft YaHei'}, children=[
    
    # 顶部：大标题 + 筛选器（组件 1）
    html.Div(style={'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'marginBottom': '20px'}, children=[
        html.H1("Superstore 零售经营数据交互大屏", style={'color': TEXT, 'margin': '0'}),
        html.Div(style={'display': 'flex', 'alignItems': 'center'}, children=[
            html.Label("选择年份：", style={'color': TEXT, 'marginRight': '10px', 'fontSize': '18px'}),
            dcc.Dropdown(
                id='year-dropdown',
                options=[{'label': '全部年份', 'value': 'All'}] + [{'label': str(y), 'value': y} for y in sorted(df['Order Year'].unique())],
                value='All', clearable=False,
                style={'width': '200px', 'color': '#000'} # 下拉框文字设为黑色
            )
        ])
    ]),

    html.Div(id='kpi-container', style={'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '20px'}),

    html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px', 'marginBottom': '20px'}, children=[
        html.Div(dcc.Graph(id='trend-chart'), style={'backgroundColor': PANEL, 'borderRadius': '10px'}),
        html.Div(dcc.Graph(id='category-pie'), style={'backgroundColor': PANEL, 'borderRadius': '10px'}),
    ]),
    html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '20px'}, children=[
        html.Div(dcc.Graph(id='discount-bar'), style={'backgroundColor': PANEL, 'borderRadius': '10px'}),
        html.Div(dcc.Graph(id='region-scatter'), style={'backgroundColor': PANEL, 'borderRadius': '10px'}),
    ])
])

@app.callback(
    [Output('kpi-container', 'children'),
     Output('trend-chart', 'figure'),
     Output('category-pie', 'figure'),
     Output('discount-bar', 'figure'),
     Output('region-scatter', 'figure')],
    [Input('year-dropdown', 'value')]
)
def update_dashboard(selected_year):
    dff = df.copy() if selected_year == 'All' else df[df['Order Year'] == selected_year]

    sales = dff['Sales'].sum()
    profit = dff['Profit'].sum()
    margin = profit / sales * 100 if sales > 0 else 0
    orders = dff['Order ID'].nunique()
    
    def create_kpi(title, value, color):
        return html.Div(style={'backgroundColor': PANEL, 'padding': '20px', 'borderRadius': '10px', 'width': '23%', 'borderTop': f'4px solid {color}'}, children=[
            html.H4(title, style={'color': TICK, 'margin': '0 0 10px 0', 'fontSize': '14px'}),
            html.H2(value, style={'color': color, 'margin': '0', 'fontSize': '28px'})
        ])
        
    kpi = [
        create_kpi("总销售额", f"${sales/1e4:,.1f} 万", SKY),
        create_kpi("总利润", f"${profit/1e4:,.1f} 万", TEAL),
        create_kpi("整体利润率", f"{margin:.1f}%", GOLD),
        create_kpi("订单总数", f"{orders:,}", BLUE)
    ]

    trend_df = dff.groupby("Order YM")[["Sales", "Profit"]].sum().reset_index()
    fig_trend = px.line(trend_df, x="Order YM", y=["Sales", "Profit"], title="月度销售与利润趋势", markers=True, color_discrete_sequence=[SKY, GOLD])
    fig_trend.update_layout(**chart_layout)

    cat_df = dff.groupby("Category")["Sales"].sum().reset_index()
    fig_pie = px.pie(cat_df, names="Category", values="Sales", hole=0.6, title="类目销售结构", color_discrete_sequence=[BLUE, SKY, TEAL])
    fig_pie.update_layout(**chart_layout)

    disc_df = dff.groupby("Discount Band")["Profit"].sum().reset_index()
    disc_df["Color"] = disc_df["Profit"].apply(lambda x: TEAL if x > 0 else RED)
    fig_disc = px.bar(disc_df, x="Discount Band", y="Profit", title="各折扣档位利润", color="Color", color_discrete_map="identity")
    fig_disc.update_layout(**chart_layout)

    reg_df = dff.groupby(["Region", "Category"]).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).reset_index()
    reg_df["Margin"] = reg_df["Profit"] / reg_df["Sales"] * 100
    fig_scatter = px.scatter(reg_df, x="Sales", y="Margin", color="Category", size="Sales", hover_data=["Region"], title="大区×类目 销售额与利润率散点", color_discrete_sequence=[BLUE, SKY, TEAL])
    fig_scatter.update_layout(**chart_layout)

    return kpi, fig_trend, fig_pie, fig_disc, fig_scatter

print("启动大屏服务器")
app.run(jupyter_mode="inline", debug=False, port=8051)

正在启动大屏服务器...
